# 압축 강도 증가에 따른 FPIR 급증 진단

목적: FIQA 성능 평가와 분리하여, 기존 완료 run에서 **어떤 검색 방식·임계값 정책에서 오수락이 증가하는지** 재현하고 사건·점수 변화로 분해합니다. 1차 구현은 기존 artifact 읽기 전용이며 FR 추론·압축기 재학습·threshold 선택을 수행하지 않습니다.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "research").is_dir())
SOURCE_MODEL = "edgeface"  # arcface / adaface / magface / edgeface
PROFILES = tuple(f"pq_512_m{m}_b8" for m in (128, 64, 32, 16, 8))
MODES = ("pq_reconstruction_cosine", "pq_one_sided_cosine", "pq_adc_exhaustive")
TARGET_FPIRS = (0.01, 0.05, 0.10, 0.20, 0.30)
FOCUS_FPIR = 0.01
SEED = 8972  # 이번 진단 CI에만 사용; 과거 실험 seed는 변경하지 않음
BOOTSTRAP_RESAMPLES = 2000
WRITE_RESULTS = True
OUTPUT_ROOT = PROJECT_ROOT / "results" / "diagnostics" / "compression_fpir"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


## 1. 비교와 해석 규칙

- FPIR: 미등록 query의 gallery 최대 점수가 임계값 이상인 비율입니다. 개별 pair FMR이 아닙니다.
- H1: 기존 승자의 점수가 올라가 임계값을 넘는가? H2: 새 후보가 최대가 되면서 추가 상승하는가?
- H3: 원본 임계값을 유지할 때와 조건별 재보정할 때 결과가 다른가? H4: 양쪽 복원과 gallery-only 압축의 양상이 다른가?
- 모든 인접 압축 단계를 비교합니다. test로 유리한 threshold/seed를 선택하지 않습니다.
- ADC는 별도 점수 공간이므로 cosine과 점수를 빼거나 원본 cosine 임계값을 적용하지 않습니다.


In [2]:
from IPython.display import display
from research.experiments.compression_fpir_failure_diagnosis import (
    SOURCE_RUNS, run_diagnosis, write_diagnosis, LIMITATIONS,
)
print("Pinned source:", PROJECT_ROOT / "runs" / SOURCE_RUNS[SOURCE_MODEL])
print("CPU artifact analysis only; no GPU inference.")
print(LIMITATIONS)


Pinned source: c:\ronbun\runs\survface_20260901\20260901-R001-56c2f3ed_step4_survface_edgeface-a348c305af33c223b337
CPU artifact analysis only; no GPU inference.
Descriptive diagnostic, not a causal identification of quantization geometry. FPIR Wilson and paired bootstrap are query-level, fixed-gallery/threshold/codec; identity dependence, calibration uncertainty and multiple comparisons are not covered. TPIR20 is genuine-score threshold AND rank<=20, not maximum-score acceptance. ADC and cosine scores are not subtracted. Recalibration is not recognition recovery. Test results must not select thresholds, seeds or compression profiles.


## 2. 출처 검증 및 재현

완료 run, v6 요약, 원장 파일 크기·SHA-256, 모델·프로토콜·query 집합을 검증합니다. 원장 TPIR20은 정답 점수 통과 AND Top-20인지 확인하고 요약 건수와 대조합니다. 파일이 없거나 불일치하면 다른 run으로 자동 대체하지 않고 중단합니다. 전체 5개 프로파일·3개 검색 방식의 기존 원장을 읽습니다.


In [3]:
result = run_diagnosis(
    PROJECT_ROOT, model=SOURCE_MODEL, profiles=PROFILES, modes=MODES,
    targets=TARGET_FPIRS, seed=SEED, resamples=BOOTSTRAP_RESAMPLES,
)
summary = result["summary"]
focus = summary.loc[summary.target_fpir == FOCUS_FPIR].copy()
display(focus[["compression_profile", "search_mode", "threshold_policy",
               "n", "reference_fa", "candidate_fa", "origin_fpir", "compressed_fpir",
               "compressed_fpir_ci_low", "compressed_fpir_ci_high", "target_met_on_test",
               "compressed_tpir20_count", "mated_count", "compressed_tpir20",
               "origin_rank20", "compressed_rank20"]])


,compression_profile,search_mode,threshold_policy,n,reference_fa,candidate_fa,origin_fpir,compressed_fpir,compressed_fpir_ci_low,compressed_fpir_ci_high,target_met_on_test,compressed_tpir20_count,mated_count,compressed_tpir20,origin_rank20,compressed_rank20
0,pq_512_m128_b8,pq_reconstruction_cosine,frozen_origin,121736,1444,398,0.011862,0.003269,0.002964,0.003606,True,0,60423,0.000000,0.182828,0.182464
1,pq_512_m128_b8,pq_reconstruction_cosine,recalibrated_compressed,121736,1444,1402,0.011862,0.011517,0.010933,0.012132,False,1,60423,0.000017,0.182828,0.182464
10,pq_512_m64_b8,pq_reconstruction_cosine,frozen_origin,121736,1444,926,0.011862,0.007607,0.007134,0.008110,True,0,60423,0.000000,0.182828,0.169621
11,pq_512_m64_b8,pq_reconstruction_cosine,recalibrated_compressed,121736,1444,1351,0.011862,0.011098,0.010525,0.011702,False,0,60423,0.000000,0.182828,0.169621
20,pq_512_m32_b8,pq_reconstruction_cosine,frozen_origin,121736,1444,2690,0.011862,0.022097,0.021286,0.022938,False,8,60423,0.000132,0.182828,0.138259
21,pq_512_m32_b8,pq_reconstruction_cosine,recalibrated_compressed,121736,1444,1376,0.011862,0.011303,0.010725,0.011913,False,2,60423,0.000033,0.182828,0.138259
30,pq_512_m16_b8,pq_reconstruction_cosine,frozen_origin,121736,1444,6674,0.011862,0.054824,0.053559,0.056116,False,59,60423,0.000976,0.182828,0.107989
31,pq_512_m16_b8,pq_reconstruction_cosine,recalibrated_compressed,121736,1444,1405,0.011862,0.011541,0.010957,0.012157,False,3,60423,0.000050,0.182828,0.107989
40,pq_512_m8_b8,pq_reconstruction_cosine,frozen_origin,121736,1444,15096,0.011862,0.124006,0.122166,0.125869,False,172,60423,0.002847,0.182828,0.082055
41,pq_512_m8_b8,pq_reconstruction_cosine,recalibrated_compressed,121736,1444,1312,0.011862,0.010777,0.010213,0.011373,False,3,60423,0.000050,0.182828,0.082055


## 3. 신규·소멸 오수락과 인접 압축 단계

`new_fa`: 원본에서는 거절, 압축 후 오수락. `lost_fa`: 반대 사건. **압축 FA = 원본 FA + 신규 − 소멸**입니다. `both_fa`, `neither_fa`를 합하면 전체 미등록 query 수와 같습니다. 아래 CI는 고정 조건의 query 단위 탐색적 paired bootstrap입니다. identity 상관·calibration 분할·codec 변동·다중 비교는 포함하지 않습니다.


In [4]:
display(focus[["compression_profile", "search_mode", "threshold_policy",
               "both_fa", "neither_fa", "new_fa", "lost_fa", "new_fa_same_winner",
               "new_fa_changed_winner", "delta_fpir", "delta_ci_low", "delta_ci_high"]])
adjacent = result["adjacent"]
display(adjacent.loc[adjacent.target_fpir == FOCUS_FPIR] if not adjacent.empty else adjacent)


,compression_profile,search_mode,threshold_policy,both_fa,neither_fa,new_fa,lost_fa,new_fa_same_winner,new_fa_changed_winner,delta_fpir,delta_ci_low,delta_ci_high
0,pq_512_m128_b8,pq_reconstruction_cosine,frozen_origin,398,120292,0,1046,0,0,-0.008592,-0.009126,-0.008083
1,pq_512_m128_b8,pq_reconstruction_cosine,recalibrated_compressed,1294,120184,108,150,87,21,-0.000345,-0.000616,-0.000082
10,pq_512_m64_b8,pq_reconstruction_cosine,frozen_origin,863,120229,63,581,23,40,-0.004255,-0.004658,-0.003836
11,pq_512_m64_b8,pq_reconstruction_cosine,recalibrated_compressed,1103,120044,248,341,98,150,-0.000764,-0.001150,-0.000378
20,pq_512_m32_b8,pq_reconstruction_cosine,frozen_origin,1337,118939,1353,107,448,905,0.010235,0.009627,0.010876
21,pq_512_m32_b8,pq_reconstruction_cosine,recalibrated_compressed,1028,119944,348,416,108,240,-0.000559,-0.001010,-0.000131
30,pq_512_m16_b8,pq_reconstruction_cosine,frozen_origin,1428,115046,5246,16,1267,3979,0.042962,0.041795,0.044145
31,pq_512_m16_b8,pq_reconstruction_cosine,recalibrated_compressed,936,119823,469,508,87,382,-0.000320,-0.000846,0.000181
40,pq_512_m8_b8,pq_reconstruction_cosine,frozen_origin,1436,106632,13660,8,1840,11820,0.112144,0.110370,0.114001
41,pq_512_m8_b8,pq_reconstruction_cosine,recalibrated_compressed,715,119695,597,729,96,501,-0.001084,-0.001709,-0.000526


,compression_profile,search_mode,threshold_policy,target_fpir,reference_profile,n,reference_fa,candidate_fa,both_fa,neither_fa,new_fa,lost_fa,delta_fpir,delta_ci_low,delta_ci_high
0,pq_512_m64_b8,pq_reconstruction_cosine,frozen_origin,0.01,pq_512_m128_b8,121736,398,926,390,120802,536,8,0.004337,0.003968,0.004723
1,pq_512_m64_b8,pq_reconstruction_cosine,recalibrated_compressed,0.01,pq_512_m128_b8,121736,1402,1351,1141,120124,210,261,-0.000419,-0.000764,-0.000082
10,pq_512_m32_b8,pq_reconstruction_cosine,frozen_origin,0.01,pq_512_m64_b8,121736,926,2690,926,119046,1764,0,0.014490,0.013792,0.015238
11,pq_512_m32_b8,pq_reconstruction_cosine,recalibrated_compressed,0.01,pq_512_m64_b8,121736,1351,1376,1150,120159,226,201,0.000205,-0.000131,0.000526
20,pq_512_m16_b8,pq_reconstruction_cosine,frozen_origin,0.01,pq_512_m32_b8,121736,2690,6674,2683,115055,3991,7,0.032727,0.031683,0.033770
21,pq_512_m16_b8,pq_reconstruction_cosine,recalibrated_compressed,0.01,pq_512_m32_b8,121736,1376,1405,1095,120050,310,281,0.000238,-0.000173,0.000641
30,pq_512_m8_b8,pq_reconstruction_cosine,frozen_origin,0.01,pq_512_m16_b8,121736,6674,15096,6522,106488,8574,152,0.069182,0.067761,0.070628
31,pq_512_m8_b8,pq_reconstruction_cosine,recalibrated_compressed,0.01,pq_512_m16_b8,121736,1405,1312,860,119879,452,545,-0.000764,-0.001281,-0.000254
40,pq_512_m64_b8,pq_one_sided_cosine,frozen_origin,0.01,pq_512_m128_b8,121736,775,791,683,120853,108,92,0.000131,-0.000099,0.000370
41,pq_512_m64_b8,pq_one_sided_cosine,recalibrated_compressed,0.01,pq_512_m128_b8,121736,1412,1348,1217,120193,131,195,-0.000526,-0.000822,-0.000238


## 4. 분포 상단과 최대 점수 변화의 분해

cosine에서 `최대 점수 변화 = 원본 승자에서의 점수 변화 + 새 후보 선택으로 얻은 점수 증가`입니다. 원본 승자가 그대로여도 오수락이 생길 수 있습니다. 빈 cohort의 평균은 0이 아니라 NaN입니다. 부동소수점 오차 허용치는 1e-6이며 작은 음의 선택 이득도 원값을 유지합니다.

분위수는 미등록 query별 최대값의 분포입니다. 원본/압축 ADC 점수는 서로 다른 척도임에 주의하세요. 이 분해는 관찰된 점수 변화의 항등식이지 양자화 기하학의 인과 규명이 아닙니다.


In [5]:
tails = result["tails"]
display(tails.loc[(tails.target_fpir == FOCUS_FPIR) & (tails["quantile"] == 0.99)])
winners = result["winners"]
display(winners.loc[(winners.target_fpir == FOCUS_FPIR) & (winners.cohort == "new_fa")] if not winners.empty else winners)
display(focus[["compression_profile", "search_mode", "threshold_policy",
               "origin_threshold", "compressed_threshold",
               "score_effect_at_origin_threshold", "threshold_effect_after_score_change",
               "compressed_tpir20", "compressed_rank20"]])


,compression_profile,search_mode,threshold_policy,target_fpir,side,quantile,score,threshold,score_space
2,pq_512_m128_b8,pq_reconstruction_cosine,frozen_origin,0.01,origin,0.99,0.932216,0.928693,cosine_similarity
7,pq_512_m128_b8,pq_reconstruction_cosine,frozen_origin,0.01,compressed,0.99,0.909543,0.928693,cosine_similarity
12,pq_512_m128_b8,pq_reconstruction_cosine,recalibrated_compressed,0.01,origin,0.99,0.932216,0.928693,cosine_similarity
17,pq_512_m128_b8,pq_reconstruction_cosine,recalibrated_compressed,0.01,compressed,0.99,0.909543,0.906232,cosine_similarity
102,pq_512_m64_b8,pq_reconstruction_cosine,frozen_origin,0.01,origin,0.99,0.932216,0.928693,cosine_similarity
107,pq_512_m64_b8,pq_reconstruction_cosine,frozen_origin,0.01,compressed,0.99,0.920938,0.928693,cosine_similarity
112,pq_512_m64_b8,pq_reconstruction_cosine,recalibrated_compressed,0.01,origin,0.99,0.932216,0.928693,cosine_similarity
117,pq_512_m64_b8,pq_reconstruction_cosine,recalibrated_compressed,0.01,compressed,0.99,0.920938,0.917829,cosine_similarity
202,pq_512_m32_b8,pq_reconstruction_cosine,frozen_origin,0.01,origin,0.99,0.932216,0.928693,cosine_similarity
207,pq_512_m32_b8,pq_reconstruction_cosine,frozen_origin,0.01,compressed,0.99,0.952465,0.928693,cosine_similarity


,compression_profile,search_mode,threshold_policy,target_fpir,cohort,count,fixed_winner_drift_mean,selection_gain_mean,maximum_score_drift_mean,decomposition_max_abs_residual,selection_gain_min_raw
1,pq_512_m128_b8,pq_reconstruction_cosine,frozen_origin,0.01,new_fa,0,NaN,NaN,NaN,0.0,-5.270312e-07
5,pq_512_m128_b8,pq_reconstruction_cosine,recalibrated_compressed,0.01,new_fa,108,-0.017518,0.000929,-0.016589,0.0,-5.270312e-07
41,pq_512_m64_b8,pq_reconstruction_cosine,frozen_origin,0.01,new_fa,63,0.004699,0.005836,0.010535,0.0,-5.967644e-07
45,pq_512_m64_b8,pq_reconstruction_cosine,recalibrated_compressed,0.01,new_fa,248,-0.001929,0.005325,0.003396,0.0,-5.967644e-07
81,pq_512_m32_b8,pq_reconstruction_cosine,frozen_origin,0.01,new_fa,1353,0.023822,0.008295,0.032117,0.0,-6.682790e-07
85,pq_512_m32_b8,pq_reconstruction_cosine,recalibrated_compressed,0.01,new_fa,348,0.033118,0.007820,0.040938,0.0,-6.682790e-07
121,pq_512_m16_b8,pq_reconstruction_cosine,frozen_origin,0.01,new_fa,5246,0.042748,0.016279,0.059027,0.0,-5.679068e-07
125,pq_512_m16_b8,pq_reconstruction_cosine,recalibrated_compressed,0.01,new_fa,469,0.052765,0.014347,0.067111,0.0,-5.679068e-07
161,pq_512_m8_b8,pq_reconstruction_cosine,frozen_origin,0.01,new_fa,13660,0.062237,0.033665,0.095902,0.0,-5.846429e-07
165,pq_512_m8_b8,pq_reconstruction_cosine,recalibrated_compressed,0.01,new_fa,597,0.071257,0.022360,0.093616,0.0,-5.846429e-07


,compression_profile,search_mode,threshold_policy,origin_threshold,compressed_threshold,score_effect_at_origin_threshold,threshold_effect_after_score_change,compressed_tpir20,compressed_rank20
0,pq_512_m128_b8,pq_reconstruction_cosine,frozen_origin,0.928693,0.928693,-0.008592,0.000000,0.000000,0.182464
1,pq_512_m128_b8,pq_reconstruction_cosine,recalibrated_compressed,0.928693,0.906232,-0.008592,0.008247,0.000017,0.182464
10,pq_512_m64_b8,pq_reconstruction_cosine,frozen_origin,0.928693,0.928693,-0.004255,0.000000,0.000000,0.169621
11,pq_512_m64_b8,pq_reconstruction_cosine,recalibrated_compressed,0.928693,0.917829,-0.004255,0.003491,0.000000,0.169621
20,pq_512_m32_b8,pq_reconstruction_cosine,frozen_origin,0.928693,0.928693,0.010235,0.000000,0.000132,0.138259
21,pq_512_m32_b8,pq_reconstruction_cosine,recalibrated_compressed,0.928693,0.949165,0.010235,-0.010794,0.000033,0.138259
30,pq_512_m16_b8,pq_reconstruction_cosine,frozen_origin,0.928693,0.928693,0.042962,0.000000,0.000976,0.107989
31,pq_512_m16_b8,pq_reconstruction_cosine,recalibrated_compressed,0.928693,0.971475,0.042962,-0.043282,0.000050,0.107989
40,pq_512_m8_b8,pq_reconstruction_cosine,frozen_origin,0.928693,0.928693,0.112144,0.000000,0.002847,0.082055
41,pq_512_m8_b8,pq_reconstruction_cosine,recalibrated_compressed,0.928693,0.988979,0.112144,-0.113229,0.000050,0.082055


## 5. 판정과 다음 단계

1. `frozen_origin`만 급증하고 재보정으로 줄어들면 점수·임계값 불일치와 일치하는 관측입니다. TPIR20/Rank20까지 회복했는지 별도로 확인하세요.
2. `new_fa_same_winner > 0`이면 후보 변경은 오수락 증가의 필요조건이 아닙니다.
3. 양쪽 복원에서만 큰 증가가 보여도 query 압축의 인과적 기여를 확정하지 않습니다. query-only 대조가 추가로 필요합니다.
4. 재보정 후에도 목표 초과가 남으면 calibration→test 전이 분석과 연결합니다. 이 노트북은 새로운 분할 안정성 실험을 수행하지 않습니다.

**후속 2차 미구현:** 정규화된 복원 오차 방향의 세 항 분해, cosine calibration 재생, query-only 대조. **후속 3차:** 여러 모델·데이터셋의 통합 보고. 모델 설정 변경은 지원하지만 모델 간 동일 cohort를 가정한 paired 비교는 하지 않습니다.

모든 rate/CI는 0~1 단위입니다. 퍼센트는 ×100, 차이의 ×100은 %p입니다. TPIR20의 새 cluster CI는 여기서 산출하지 않습니다.


In [6]:
if WRITE_RESULTS:
    output_dir = write_diagnosis(result, OUTPUT_ROOT)
    print("New diagnostic artifacts:", output_dir)
else:
    print("Read-only: no diagnostic output written. Set WRITE_RESULTS=True to export.")
display({k: v for k, v in result["provenance"].items() if k != "inventory"})


New diagnostic artifacts: c:\ronbun\results\diagnostics\compression_fpir\diagnostic-93ab3aaa203a46a7


{'model': 'edgeface',
 'source_run': 'C:\\ronbun\\runs\\survface_20260901\\20260901-R001-56c2f3ed_step4_survface_edgeface-a348c305af33c223b337',
 'run_id': '20260901-R001-56c2f3ed',
 'generated_at_utc': '2026-09-16T02:05:58.221939+00:00',
 'run_manifest_sha256': '6f4f967c42c5617fb2d9057adce195e95f4c13d5b31d918c326d2534f401c9e3',
 'summary_manifest_sha256': 'ca0a4423446076dc7f6cb42af2b90898d2f7d65d5379cc2c7ed4474cd1d1916a',
 'ledger_manifest_sha256': '23ce579fd96b0c0ea6bcd7df68072086b671de4736a0f01ac16f4a7f4bb49eca',
 'implementation_sha256': '8493b3d8437473f4f673e0e607f38928c213e0bf4e0812a0917f08ebc08a5a1d',
 'metrics_sha256': 'bae78f0349a68c086de795d47553876243271e6a210bff2664ee223485c2e64d',
 'search_conditions_sha256': 'a1c7abecb8c029f48bdf625a586de0e6cea9aac6d8cef6be01938f77c1829796',
 'numpy_version': '1.26.4',
 'pandas_version': '3.0.3',
 'source_evaluator_git': {'commit': '9627e9f33b15256aaaf77e4731f88833c3eff89f',
  'branch': 'step10',
  'dirty': False,
  'working_tree_diff_sha